# CIC-IDS-2017 Multi-Class XGBoost Anomaly Detector

This notebook loads the `cic-collection.parquet` dataset, preprocesses the network flow features (handling NaNs, Infinity, and whitespaces), and trains a Multi-Class XGBoost classifier to distinguish between BENIGN, DDoS, PortScan, and Botnet traffic.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Ensure model directory exists
os.makedirs(os.path.join("..", "models"), exist_ok=True)

## 1. Load Data
We load the `cic-collection.parquet` dataset.

In [ ]:
data_path = os.path.join("..", "data", "raw", "cic-collection.parquet")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "raw", "cic-collection.parquet")

df = pd.read_parquet(data_path)
print(f"Loaded dataset with shape: {df.shape}")

## 2. Preprocessing
- Clean column names (strip whitespaces)
- Handle `np.inf` and `np.nan` values which are common in calculated flow rates.
- Filter for target classes (Benign, DDoS, Portscan, Botnet)
- Map classes to integers (0-3).

In [ ]:
# Strip whitespace from column names
df.columns = df.columns.str.strip()

print("Class distribution before filtering:")
print(df['Label'].value_counts())

TARGET_CLASSES = {
    'Benign': 0,
    'DDoS': 1,
    'Portscan': 2,
    'Botnet': 3
}

# Filter dataset to keep only the target classes
df = df[df['Label'].isin(TARGET_CLASSES.keys())].copy()

# Encode Label
df['Label'] = df['Label'].map(TARGET_CLASSES)

print("\nClass distribution after mapping:")
print(df['Label'].value_counts())

# Drop non-numeric columns if present (e.g., IPs, Timestamp, Flow ID)
cols_to_drop = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Convert all remaining columns to numeric (coerce errors to NaN)
for col in df.columns:
    if col != 'Label':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Handle inf and NaN values
df = df.replace([np.inf, -np.inf], np.nan)
df = df.fillna(0)

print(f"\nDataset shape after preprocessing: {df.shape}")

## 3. Train/Test Split
Split the data into 80% training and 20% testing sets.

In [ ]:
X = df.drop(columns=['Label'])
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 4. Train Multi-Class XGBoost Classifier
Initialize and train the XGBoost model for multi-class classification.

In [ ]:
model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='mlogloss'
)

print("Training multi-class model...")
model.fit(X_train, y_train)
print("Training complete.")

## 5. Evaluation
Evaluate model performance on the test set using Confusion Matrix and Classification Report, and plot feature importance.

In [ ]:
y_pred = model.predict(X_test)
target_names = ['Benign', 'DDoS', 'Portscan', 'Botnet']

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# Feature Importance Plot
fig, ax = plt.subplots(figsize=(10, 8))
xgb.plot_importance(model, ax=ax, max_num_features=15, height=0.5, 
                    title='Top 15 Feature Importances')
plt.savefig('feature_importance.png')
plt.show()

## 6. Save Model
Serialize the trained multi-class model for inference.

In [ ]:
model_path = os.path.join("..", "models", "xgboost_anomaly_detector.pkl")
if not os.path.exists(os.path.dirname(model_path)):
    model_path = os.path.join("models", "xgboost_anomaly_detector.pkl")

joblib.dump(model, model_path)
print(f"Multi-class model successfully saved to {model_path}")